# 09 — Codebook draft (themes 15 / topics 150, full cut)

**Operating point** (2026-08-22, following the researcher-approved rule
that stability reports and never gates): topics **K=150** — best
topic-scale stability (0.474), fewest cohesion flags, and granularity
closest to the v0 benchmark that set the CAP-agreement bar — and themes
**K=15**. **Every** topic cluster is drafted, not only the stable ones;
each entry carries its stability and cohesion diagnostics for the
ratification pass in notebook 10.

**Discipline** (as in the superseded pooled run, unchanged): the LLM sees
only member phrases and neighbor phrases; exemplars must be verbatim
members (parser drops inventions); labels pass the directional lint; every
call is logged with its prompt version.

**Inputs.** `interim/topics/taxonomy/` (08); Qwen/Qwen3.5-4B.

**Outputs.** `taxonomy/codebook_draft.parquet`;
`quality_reports/20260822_codebook_draft_t1.md`;
`taxonomy/status/codebook_draft.jsonl`.

## Setup — cut, diagnostics, and topic→theme nesting

In [1]:
import hashlib
import json as _json
import os
import re
import sys
import time

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

from paths import HF_CACHE, ROOT, topics_run_dir

os.environ["HF_HOME"] = str(HF_CACHE)
K_TOPIC, K_THEME = 150, 15
TAX = topics_run_dir("taxonomy")

phrases = pd.read_parquet(TAX / "phrases.parquet")
cuts = pd.read_parquet(TAX / "cut_labels.parquet")
stats = pd.read_parquet(TAX / "cluster_stats.parquet")

ph = phrases[phrases["family_id"] >= 0].merge(
    cuts[["family_id", f"K{K_TOPIC}", f"K{K_THEME}"]], on="family_id")
ph = ph.rename(columns={f"K{K_TOPIC}": "topic_c", f"K{K_THEME}": "theme_c"})
stab_t = stats.query("K == @K_TOPIC").set_index("cluster")
stab_h = stats.query("K == @K_THEME").set_index("cluster")

nest = (ph.groupby(["topic_c", "theme_c"])["weight"].sum().reset_index()
          .sort_values("weight", ascending=False)
          .drop_duplicates("topic_c").set_index("topic_c"))
share = nest["weight"] / ph.groupby("topic_c")["weight"].sum()
print(f"{K_TOPIC} topics / {K_THEME} themes; topic->theme plurality "
      f"share: median {share.median():.0%}, min {share.min():.0%}")

150 topics / 15 themes; topic->theme plurality share: median 77%, min 19%


## Drafting — all topics, then all themes

Same prompt (`codebook-draft-0.1`) and verbatim-exemplar parser as
validated before; per-cluster seeds for bit reproducibility.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import stance
import summarize as S

MODEL = "Qwen/Qwen3.5-4B"
CODEBOOK_PROMPT_VERSION = "codebook-draft-0.1"
MAX_NEW_TOKENS = 512

CB_SYSTEM = ("You are helping construct a topic codebook for the analysis "
             "of United States congressional hearings. You use only the "
             "phrases provided; you never invent content.")

CB_USER = """Below are the member phrases of one topic cluster (most frequent first), followed by the top phrases of its nearest neighboring clusters.

CLUSTER MEMBERS (phrase — mentions):
{members}

NEIGHBOR 1 (a different topic): {n1}
NEIGHBOR 2 (a different topic): {n2}

Task: draft the codebook entry for this cluster.
- "label": a short, specific, neutral noun phrase naming the topic. Never directional wording (no ban/support/expand/protect); name the policy object, not a stance toward it.
- "definition": one sentence stating what belongs in this topic.
- "exclusion_notes": one or two short notes on how this topic differs from the two neighbors shown.
- "exemplars": exactly 5 phrases copied VERBATIM from the CLUSTER MEMBERS list above (or all of them, if fewer than 5 are listed).

Return only a JSON object with exactly those four fields. Your very first output character must be {{."""

def parse_entry(output, members_cf):
    errors = []
    text = output.strip()
    if text.startswith("```"):
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = _json.loads(text)
    except _json.JSONDecodeError:
        return None, errors + ["json_parse"]
    if not isinstance(raw, dict):
        return None, errors + ["not_object"]
    label = str(raw.get("label", "")).strip()
    definition = str(raw.get("definition", "")).strip()
    notes = raw.get("exclusion_notes", [])
    if isinstance(notes, str):
        notes = [notes]
    ex = [str(e).strip() for e in raw.get("exemplars", [])]
    verbatim = [e for e in ex if e.casefold() in members_cf]
    if len(verbatim) < len(ex):
        errors.append(f"exemplars_not_verbatim:{len(ex) - len(verbatim)}")
    if not label or not definition:
        return None, errors + ["missing_field"]
    return {"label": label, "definition": definition,
            "exclusion_notes": [str(x) for x in notes][:2],
            "exemplars": verbatim[:5],
            "label_lint": stance.lint_topic(label)}, errors

dat = np.load(TAX / "unit_embeddings.npz", allow_pickle=True)
X = dat["X"].astype(np.float32)
nk = {n: i for i, n in enumerate(dat["norm_key"])}
ph["ux"] = ph["norm"].str.casefold().map(nk)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto",
                                             device_map="cuda")
model.eval()

def centroid(g):
    v = (X[g["ux"].to_numpy()] * g["weight"].to_numpy()[:, None]).sum(0)
    return v / np.linalg.norm(v)

def top_phrases(g, n):
    return g.groupby("phrase")["weight"].sum().nlargest(n)

LOG = TAX / "status" / "codebook_draft.jsonl"
rows = []
for kind, K, col, st in (("topic", K_TOPIC, "topic_c", stab_t),
                         ("theme", K_THEME, "theme_c", stab_h)):
    groups = {c: ph[ph[col] == c] for c in range(1, K + 1)}
    groups = {c: g for c, g in groups.items() if len(g)}
    ids = list(groups)
    C = np.stack([centroid(g) for g in groups.values()])
    sims = C @ C.T
    np.fill_diagonal(sims, -1)
    for i, c in enumerate(ids):
        nb_ids = [ids[j] for j in np.argsort(sims[i])[::-1][:2]]
        neighbors = [list(top_phrases(groups[nn], 8).index)
                     for nn in nb_ids]
        torch.manual_seed(int.from_bytes(hashlib.sha256(
            f"{kind}:{c}".encode()).digest()[:4], "big"))
        tp = top_phrases(groups[c], 30)
        members = "\n".join(f"[{j}] {p} — {int(w)}"
                             for j, (p, w) in enumerate(tp.items(), 1))
        user = CB_USER.format(members=members,
                              n1="; ".join(neighbors[0][:8]),
                              n2="; ".join(neighbors[1][:8]))
        log = S.CallLog(LOG, f"{kind}:{c}", MODEL,
                        prompt_version=CODEBOOK_PROMPT_VERSION)
        t0 = time.time()
        out, hit_cap = S._generate(model, tokenizer, CB_SYSTEM, user,
                                   max_new_tokens=MAX_NEW_TOKENS)
        log.write("codebook", 0, 0, 1, user, out, tokenizer,
                  time.time() - t0)
        entry, errors = parse_entry(out, {p.casefold() for p in tp.index})
        if hit_cap:
            errors.append("hit_token_cap")
        srow = st.loc[c] if c in st.index else None
        rows.append({
            "kind": kind, "cluster": int(c),
            "stability": float(srow["stability"]) if srow is not None else np.nan,
            "cohesion": float(srow["cohesion"]) if srow is not None else np.nan,
            "weight": float(groups[c]["weight"].sum()),
            "n_phrases": int(groups[c]["phrase"].nunique()),
            "top_members": _json.dumps(list(tp.index[:10])),
            "entry": _json.dumps(entry) if entry else None,
            "errors": ";".join(errors),
        })
    print(f"{kind}: {len(ids)} entries drafted")

draft_df = pd.DataFrame(rows)
draft_df.to_parquet(TAX / "codebook_draft.parquet", index=False)
ok = draft_df["entry"].notna()
print(f"clean {ok.sum()}/{len(draft_df)}; "
      f"errors on {(draft_df['errors'] != '').sum()}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

topic: 150 entries drafted


theme: 15 entries drafted
clean 165/165; errors on 5


## Ratification document (all topics under plurality themes)

In [3]:
themes = draft_df[draft_df["kind"] == "theme"].set_index("cluster")
topics = draft_df[draft_df["kind"] == "topic"]
tt = {int(t): int(nest.loc[t, "theme_c"]) for t in nest.index}

lines = ["# Codebook draft (Track-1, themes 15 / topics 150) — for ratification",
         "", "Stability and cohesion are diagnostics; the ratification "
         "decides KEEP / MERGE / DROP per entry.", ""]
for th in sorted(themes.index):
    e = (_json.loads(themes.loc[th, "entry"])
         if themes.loc[th, "entry"] else None)
    lines += [f"## THEME {th}: {e['label'] if e else '(draft failed)'}", ""]
    for r in topics.itertuples():
        if tt.get(r.cluster) != th:
            continue
        t = _json.loads(r.entry) if r.entry else None
        lines += [f"### Topic {r.cluster}: "
                  f"{t['label'] if t else '(failed: ' + r.errors + ')'}",
                  f"- stab {r.stability:.2f} | coh {r.cohesion:.2f} | "
                  f"w {r.weight:.0f} | {r.n_phrases} phrases"]
        if t:
            lines += [f"- def: {t['definition']}",
                      f"- ex: " + "; ".join(t["exemplars"])]
        lines += [f"- members: "
                  + "; ".join(_json.loads(r.top_members)[:8]), ""]
out = ROOT / "quality_reports" / "20260822_codebook_draft_t1.md"
out.write_text("\n".join(lines))
print(f"ratification document: {out}")

ratification document: /project2/njia_1089/acroh/projects/llm_method/quality_reports/20260822_codebook_draft_t1.md


## Verification

In [4]:
ok = draft_df["entry"].notna()
print(f"1) drafted : {ok.sum()}/{len(draft_df)}")
print(f"2) lint    : "
      f"{sum(bool(_json.loads(r.entry).get('label_lint')) for r in draft_df[ok].itertuples())} flags")
print(f"3) NEXT    : ratification + freeze + assignment (10)")

1) drafted : 165/165
2) lint    : 7 flags
3) NEXT    : ratification + freeze + assignment (10)
